In [13]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from scipy.stats import zscore
from scipy.stats import linregress
from sklearn.mixture import GaussianMixture
from scipy.stats import mode

In [14]:
def _compute_window_band_powers(vals, fs, window_size, bands, center=True, pad=True, window_type='hann'):
    n = len(vals)
    if n == 0:
        return np.zeros((0, len(bands)))

    # 窓関数
    if window_type in ['hann', 'hanning']:
        win = np.hanning(window_size)
    else:
        win = np.ones(window_size)

    # 周波数軸（必ず window_size を使う）
    freqs = np.fft.rfftfreq(window_size, d=1.0/fs)

    band_mat = np.zeros((n, len(bands)), dtype=float)

    for i in range(n):
        # ウィンドウ位置決定
        if center:
            start = i - window_size // 2
        else:
            start = i
        end = start + window_size

        # 切り出し＋パディング
        if start < 0 or end > n:
            if pad:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                left = s_idx - start
                right = end - e_idx
                seg_padded = np.pad(seg, (max(0, left), max(0, right)), mode='reflect')

                if len(seg_padded) < window_size:
                    seg_padded = np.pad(seg_padded, (0, window_size - len(seg_padded)), mode='edge')

                windowed = seg_padded[:window_size]
            else:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                if len(seg) < window_size:
                    windowed = np.pad(seg, (0, window_size - len(seg)), mode='constant')
                else:
                    windowed = seg[:window_size]
        else:
            windowed = vals[start:end]

        # DC成分除去
        windowed = windowed - np.mean(windowed)

        # 窓関数適用
        windowed = windowed * win

        # --- FFT ---
        fft_vals = np.fft.rfft(windowed)

        # 振幅（正規化：2/N）
        amp = (2.0 / window_size) * np.abs(fft_vals)

        # パワー
        power = amp ** 2

        # --- バンドごとに合計 ---
        for b_i, (f_lo, f_hi) in enumerate(bands):
            mask = (freqs >= f_lo) & (freqs < f_hi)
            band_mat[i, b_i] = float(np.sum(power[mask])) if np.any(mask) else 0.0

    return band_mat


# 加速度前処理（特徴量作成）
def preprocess_acc_sensor_minimal(df_raw, sensor_suffix="acc1",
                                  fs=100.0, window_size=1024,
                                  bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
                                  center=True, pad=True):

    df = df_raw.copy()

    # ArriveTime → time（Burst の NaN は ffill）
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce")
    df["ArriveTime"] = df["ArriveTime"].ffill()
    df["time"] = df["ArriveTime"]

    # 加速度を float に
    ax = pd.to_numeric(df.get("AccelerationX", 0), errors="coerce").fillna(0.0).values.astype(float)
    ay = pd.to_numeric(df.get("AccelerationY", 0), errors="coerce").fillna(0.0).values.astype(float)
    az = pd.to_numeric(df.get("AccelerationZ", 0), errors="coerce").fillna(0.0).values.astype(float)

    out = pd.DataFrame(index=df.index)

    # 基本列
    out[f"time_{sensor_suffix}"] = df["time"].values
    out[f"acc_x_{sensor_suffix}"] = ax
    out[f"acc_y_{sensor_suffix}"] = ay
    out[f"acc_z_{sensor_suffix}"] = az

    mag = np.sqrt(ax**2 + ay**2 + az**2)
    out[f"acc_mag_{sensor_suffix}"] = mag

    # 差分
    out[f"acc_mag_diff_{sensor_suffix}"] = np.concatenate(([0.0], np.diff(mag)))
    out[f"acc_x_diff_{sensor_suffix}"] = np.concatenate(([0.0], np.diff(ax)))
    out[f"acc_y_diff_{sensor_suffix}"] = np.concatenate(([0.0], np.diff(ay)))
    out[f"acc_z_diff_{sensor_suffix}"] = np.concatenate(([0.0], np.diff(az)))

    # バンドパワー（x/y/z/mag）
    band_x = _compute_window_band_powers(ax, fs, window_size, bands, center, pad)
    band_y = _compute_window_band_powers(ay, fs, window_size, bands, center, pad)
    band_z = _compute_window_band_powers(az, fs, window_size, bands, center, pad)
    band_m = _compute_window_band_powers(mag, fs, window_size, bands, center, pad)

    for b_i, (f_lo, f_hi) in enumerate(bands):
        band_idx = b_i + 1
        out[f"acc_x_band{band_idx}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_x[:, b_i]
        out[f"acc_y_band{band_idx}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_y[:, b_i]
        out[f"acc_z_band{band_idx}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_z[:, b_i]
        out[f"acc_mag_band{band_idx}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_m[:, b_i]

    return out


# # --- ビーコン特徴量生成のためのヘルパー関数（追加） ---
# def _compute_window_features(df_vals, window_size, center=True, method='mean'):
#     n = len(df_vals)
#     n_cols = df_vals.shape[1]
    
#     # 結果格納用 DataFrame
#     result_df = pd.DataFrame(index=df_vals.index)

#     for i in range(n):
#         # ウィンドウ位置決定
#         if center:
#             start = i - window_size // 2
#         else:
#             start = i
#         end = start + window_size

#         s_idx = max(start, 0)
#         e_idx = min(end, n)
        
#         # ウィンドウサイズが足りない場合はスキップ
#         if e_idx - s_idx < 1:
#             continue
            
#         seg_df = df_vals.iloc[s_idx:e_idx]
        
#         # (1) 各中継器のウィンドウ内統計量
#         if method == 'stats':
#             result_df.loc[i, df_vals.columns + '_mean'] = seg_df.mean()
#             result_df.loc[i, df_vals.columns + '_std'] = seg_df.std().fillna(0.0)
            
#             # 終端 - 始点
#             if len(seg_df) >= 2:
#                 result_df.loc[i, df_vals.columns + '_end_start_diff'] = seg_df.iloc[-1] - seg_df.iloc[0]
#             else:
#                 result_df.loc[i, df_vals.columns + '_end_start_diff'] = 0.0

#         # (2) 最強/2番目の最高強度
#         elif method == 'max_powers':
#             max_rssi = seg_df.max(axis=1) # ウィンドウ内での各タイムステップの最強RSSI
            
#             # ウィンドウ全体での最高強度
#             result_df.loc[i, 'max_beacon_power'] = max_rssi.max() if not max_rssi.empty else 0.0
            
#             # 2番目の最高強度（各行のRSSIをソートし、2番目の値を取得）
#             second_max_rssi = seg_df.apply(lambda row: row.nlargest(2).iloc[-1] if len(row.dropna()) >= 2 else np.nan, axis=1)
#             result_df.loc[i, 'second_max_beacon_power'] = second_max_rssi.max() if not second_max_rssi.empty and not second_max_rssi.isnull().all() else 0.0

#         # (3/4) PCA/GMM
#         elif method == 'pca_gmm':
#             # 4次元ベクトルデータ (最初の4列を使用)
#             data_4d = seg_df.iloc[:, :min(4, n_cols)].copy().dropna()
            
#             # ビーコンのRSSIは負の値が多く、0は欠損補完の場合があるため、欠損値以外のデータが少ない場合はスキップ
#             if data_4d.shape[0] >= 2 and data_4d.shape[1] == 4 and np.var(data_4d.values.flatten()) > 1e-6:
                
#                 # データを標準化
#                 data_scaled = zscore(data_4d, ddof=1)
                
#                 # PCA (上位2成分)
#                 pca = PCA(n_components=2)
#                 pca.fit(data_scaled)
#                 result_df.loc[i, 'pca_comp1'] = pca.explained_variance_ratio_[0]
#                 result_df.loc[i, 'pca_comp2'] = pca.explained_variance_ratio_[1]
                
#                 # GMM (クラスタ確率)
#                 try:
#                     gmm = GaussianMixture(n_components=3, random_state=0, max_iter=100)
#                     gmm.fit(data_scaled)
                    
#                     # GMMのクラスタ確率 (ウィンドウの最終点での確率を計算)
#                     proba = gmm.predict_proba(data_scaled.tail(1))
#                     result_df.loc[i, ['gmm_prob_c1', 'gmm_prob_c2', 'gmm_prob_c3']] = proba[0]
#                 except ValueError:
#                      result_df.loc[i, ['gmm_prob_c1', 'gmm_prob_c2', 'gmm_prob_c3']] = 0.0
#             else:
#                 result_df.loc[i, ['pca_comp1', 'pca_comp2', 'gmm_prob_c1', 'gmm_prob_c2', 'gmm_prob_c3']] = 0.0
            
#     # インデックスをリセットし、元のインデックスに合わせて0埋め
#     return result_df.sort_index().reindex(df_vals.index).fillna(0.0)


# # ビーコン前処理（特徴量作成）
# def preprocess_beacon_sensor(df_raw, window_size=512, center=True):
#     """
#     ビーコンデータ（RSSI）からウィンドウベースの特徴量を生成する。
#     """
#     df = df_raw.copy()
    
#     # ビーコンRSSI列を抽出 (time, timestamp, index 以外の列)
#     beacon_rssi_cols = [c for c in df.columns if c not in ("index", "timestamp", "time")]
#     # 注: load_and_process_by_keywordでrawデータを読み込む際、time列以外はRSSI列のはずです。
#     df_rssi = df[beacon_rssi_cols].fillna(0.0)
    
#     if df_rssi.empty:
#         return pd.DataFrame(index=df.index)

#     # (1) 各中継器の統計量
#     stats_df = _compute_window_features(df_rssi, window_size, center, method='stats')
    
#     # (2) 最強/2番目の最高強度
#     max_powers_df = _compute_window_features(df_rssi, window_size, center, method='max_powers')
    
#     # (3 & 4) PCA/GMM
#     pca_gmm_df = _compute_window_features(df_rssi, window_size, center, method='pca_gmm')

#     # 全ての特徴量を結合
#     out = pd.concat([stats_df, max_powers_df, pca_gmm_df], axis=1)
    
#     # 元の time 列を追加（マージ用）
#     out['time'] = df['time'].values
    
#     # カラム名に接尾辞を追加
#     out = out.add_suffix('_beacon')
    
#     return out.fillna(0.0)

# 既存の preprocess_beacon_sensor_minimal は削除（または上書き）
# def preprocess_beacon_sensor_minimal(df, window_size=1024):
#     df_row = df.copy()



# ---- GMM で１サンプル単位の位置クラスタ付与 ----
def assign_gmm_clusters(df, rssi_cols, n_clusters=7):
    X = df[rssi_cols].fillna(0.0).values
    gmm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=0)
    gmm.fit(X)

    df['gmm_cluster'] = gmm.predict(X)
    df[[f'gmm_prob_{k}' for k in range(n_clusters)]] = gmm.predict_proba(X)

    return df, gmm


# ---- ウィンドウ統計量 ----
def compute_window_stats(df_vals, window_size, center=True):
    n = len(df_vals)
    out = pd.DataFrame(index=range(n))  # ← 修正ポイント

    for i in range(n):
        if center:
            s = max(0, i - window_size//2)
            e = min(n, i + window_size//2)
        else:
            s = i
            e = min(n, i+window_size)

        seg = df_vals.iloc[s:e]

        out.loc[i, df_vals.columns + "_mean"] = seg.mean().values
        out.loc[i, df_vals.columns + "_std"]  = seg.std().fillna(0).values
        out.loc[i, df_vals.columns + "_max"]  = seg.max().values
        out.loc[i, df_vals.columns + "_min"]  = seg.min().values
        out.loc[i, df_vals.columns + "_range"] = (seg.max() - seg.min()).values

    return out.fillna(0.0)

# ---- GMM 特徴量生成（one-hot または 確率出力）----
def compute_gmm_features(df, n_clusters, mode="onehot"):
    """
    mode: "onehot" または "prob"
    """

    if 'gmm_cluster' not in df.columns:
        raise ValueError("df に gmm_cluster がありません。GMM を先に実行してください。")

    if mode not in ["onehot", "prob"]:
        raise ValueError("mode は 'onehot' か 'prob' を指定してください")

    out = pd.DataFrame(index=df.index)

    # ---- one-hot の場合 ----
    if mode == "onehot":
        for k in range(n_clusters):
            out[f"gmm_feat_{k}"] = (df['gmm_cluster'] == k).astype(float)

    # ---- 確率出力（predict_proba）の場合 ----
    elif mode == "prob":
        prob_cols = [f"gmm_prob_{k}" for k in range(n_clusters)]
        out[prob_cols] = df[prob_cols].values

        # 列名統一（gmm_feat_0 など）
        out.columns = [f"gmm_feat_{k}" for k in range(n_clusters)]

    return out

# ---- ウィンドウ GMM one-hot （修正版）----
def compute_window_gmm_onehot(df, n_clusters, window_size, center=True):
    n = len(df)
    out = pd.DataFrame(index=df.index)  # ← df.indexを使う

    for i, idx in enumerate(df.index):  # ← enumerateで対応づけ
        if center:
            s = max(0, i - window_size//2)
            e = min(n, i + window_size//2)
        else:
            s = i
            e = min(n, i+window_size)

        seg_labels = df['gmm_cluster'].iloc[s:e]

        if len(seg_labels) == 0:
            for k in range(n_clusters):
                out.loc[idx, f'gmm_onehot_{k}'] = 0
            continue

        m = mode(seg_labels, keepdims=False).mode
        for k in range(n_clusters):
            out.loc[idx, f'gmm_onehot_{k}'] = 1 if m == k else 0
            
    return out.fillna(0.0)


# ---- 生RSSI + 特徴量を保持する新版 preprocess_beacon ----
def preprocess_beacon(df_raw, rssi_cols=None, window_size=512, n_clusters=7, center=True):

    df = df_raw.copy()

    if rssi_cols is None:
        rssi_cols = ["810257F7", "81025919", "81025B89", "810B3B76"]

    # 1) GMM クラスタリング（1サンプル単位）
    # df, gmm = assign_gmm_clusters(df, rssi_cols, n_clusters=n_clusters)

    # 2) RSSI のウィンドウ統計量
    stats_df = compute_window_stats(df[rssi_cols], window_size, center=center)

    # 3) GMM の one-hot
    # gmm_onehot_df = compute_window_gmm_onehot(df, n_clusters, window_size, center=center)
    # gmm_onehot_df = compute_gmm_features(df, n_clusters, mode="onehot")
    
    # -------------------------
    # ★ 4) 生データを残す（重要）
    # -------------------------
    raw_df = df[["time"] + rssi_cols].rename(columns={"time": "time_beacon"})

    # -------------------------
    # ★ 5) 生 + 特徴量 の全部入れ
    # -------------------------
    # out = pd.concat([raw_df, stats_df, gmm_onehot_df], axis=1)
    out = pd.concat([raw_df, stats_df], axis=1)

    return out.fillna(0.0)



In [15]:
def load_and_process_by_keyword(keyword,
                                root_folder=r"../../../../../デスクトップ/実験",
                                save_path=None,
                                fs=100.0,
                                window_size=1024, # 加速度のウィンドウサイズ
                                bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
                                center=True,
                                pad=True,
                                interpolate_missing=True,
                                beacon_window=128): # ★ ビーコンのウィンドウサイズを引数に追加

    # 1. ファイル特定と読み込み
    # ----------------------------------------
    keyword = str(keyword)
    
    # ★ フォルダパス（旧ロジック維持）
    acc_path = os.path.join(root_folder, "data_acc")
    beacon_path = os.path.join(root_folder, "data_beacon")

    acc_all = glob.glob(os.path.join(acc_path, "*.csv"))
    beacon_all = glob.glob(os.path.join(beacon_path, "*.csv"))

    # キーワードを含むファイル特定（旧ロジック維持）
    acc_files = [f for f in acc_all if keyword.lower() in os.path.basename(f).lower()]
    beacon_files = [f for f in beacon_all if keyword.lower() in os.path.basename(f).lower()]

    if len(acc_files) < 2:
        raise FileNotFoundError("acc ファイルが2つ必要です")
    if len(beacon_files) < 1:
        raise FileNotFoundError("beacon ファイルが必要です")

    # acc1/acc2 を決める（到着時刻の最初が早い方を acc1）
    acc_files_with_min = []
    for f in acc_files:
        try:
            tmp = pd.read_csv(f)
            # ArriveTime 処理は元のロジックに従う
            tmp["ArriveTime"] = pd.to_datetime(tmp.get("ArriveTime"), errors="coerce").ffill()
            if tmp["ArriveTime"].notna().any():
                acc_files_with_min.append((f, tmp["ArriveTime"].min()))
        except Exception as e:
            print(f"Warning: Could not process file {os.path.basename(f)}. Error: {e}")
            continue

    if len(acc_files_with_min) < 2:
        raise ValueError("有効な Acc ファイルが2つ見つかりませんでした。")
        
    acc_files_with_min = sorted(acc_files_with_min, key=lambda x: x[1])
    acc1_file = acc_files_with_min[0][0]
    acc2_file = acc_files_with_min[1][0]
    beacon_file = beacon_files[0]

    print("-------", keyword, "-------")
    print("✔ acc1 →", os.path.basename(acc1_file))
    print("✔ acc2 →", os.path.basename(acc2_file))
    print("✔ beacon →", os.path.basename(beacon_file))

    # acc 読み込み（Burst対応）
    def _load_acc(path):
        tmp = pd.read_csv(path)
        # カラム名の空白をストリップ
        tmp.columns = tmp.columns.str.strip()
        
        tmp["ArriveTime"] = pd.to_datetime(tmp.get("ArriveTime"), errors="coerce").ffill()
        tmp["time"] = tmp["ArriveTime"]
        
        # 旧ロジック：欠損値のないことが保証されているため、fillna(0.0)は動作する
        # ただし、安全のため df.get ではなく列の存在チェックを推奨（今回は旧ロジック維持）
        tmp["AccelerationX"] = pd.to_numeric(tmp.get("AccelerationX"), errors="coerce").fillna(0.0)
        tmp["AccelerationY"] = pd.to_numeric(tmp.get("AccelerationY"), errors="coerce").fillna(0.0)
        tmp["AccelerationZ"] = pd.to_numeric(tmp.get("AccelerationZ"), errors="coerce").fillna(0.0)
        return tmp

    acc1_raw = _load_acc(acc1_file)
    acc2_raw = _load_acc(acc2_file)

    # beacon データ準備
    beacon_raw = pd.read_csv(beacon_file)
    if "timestamp" not in beacon_raw.columns:
        raise KeyError("beacon に timestamp がありません")

    beacon_raw["timestamp"] = pd.to_datetime(beacon_raw["timestamp"], errors="coerce")
    beacon_raw["time"] = beacon_raw["timestamp"]
    beacon_rssi_cols = [c for c in beacon_raw.columns if c not in ("index","timestamp","time")]
    # beacon_df は特徴量計算の入力として、時間とRSSI列のみを含む（旧ロジック維持）
    beacon_df = beacon_raw[["time"] + beacon_rssi_cols].sort_values("time").reset_index(drop=True) 

    # 2. 特徴量計算
    # ----------------------------------------
    # 加速度特徴量計算
    acc1_feat = preprocess_acc_sensor_minimal(acc1_raw, "acc1", fs, window_size, bands, center, pad)
    acc2_feat = preprocess_acc_sensor_minimal(acc2_raw, "acc2", fs, window_size, bands, center, pad)
    
    # ★ ビーコン特徴量計算
    # preprocess_beacon_sensor は、このコードの前に定義されている前提
    # beacon_feat = preprocess_beacon_sensor(beacon_df, window_size=beacon_window, center=center)
    beacon_feat = preprocess_beacon(beacon_df, rssi_cols=beacon_rssi_cols, window_size=beacon_window, n_clusters=7, center=center)

    beacon_feat = beacon_feat.rename(columns={"time": "time_beacon"})

    # 3. マージ処理
    # ----------------------------------------
    acc1_feat = acc1_feat.dropna(subset=["time_acc1"]).sort_values("time_acc1").reset_index(drop=True)
    acc2_feat = acc2_feat.dropna(subset=["time_acc2"]).sort_values("time_acc2").reset_index(drop=True)
    beacon_feat = beacon_feat.dropna(subset=["time_beacon"]).sort_values("time_beacon").reset_index(drop=True) # ★ beacon_feat をソート
    


    # merge_asof 用に time 名を統一
    acc1_for_merge = acc1_feat.rename(columns={"time_acc1": "time"})
    acc2_for_merge = acc2_feat.rename(columns={"time_acc2": "time"})
    beacon_for_merge = beacon_feat.rename(columns={"time_beacon": "time"}).sort_values("time") # ★ beacon_feat の時間列を 'time' にリネーム

    merged_acc = pd.merge_asof(
        acc1_for_merge,
        acc2_for_merge,
        on="time",
        direction="nearest",
        suffixes=("_acc1", "_acc2")
    )

    # ★ ビーコンの「特徴量」とマージ
    merged = pd.merge_asof(
        merged_acc.sort_values("time"),
        beacon_for_merge, # ★ beacon_df ではなく、特徴量である beacon_for_merge を使用
        on="time",
        direction="nearest"
    )

    # time_acc1 に戻す（旧ロジック維持）
    merged = merged.rename(columns={"time": "time_acc1"})

    # 4. 最終処理と返却
    # ----------------------------------------
    # 最終的な特徴量列の選択ロジックを修正
    acc_feat_cols = [c for c in merged_acc.columns if c not in ('time', 'time_acc1', 'time_acc2')]
    beacon_feat_cols = [c for c in beacon_for_merge.columns if c != 'time']
    
    # 'time_acc1' はマージ後の時間軸として保持（ただし最終的な drop がある）
    keep_cols = ['time_acc1'] + acc_feat_cols + beacon_feat_cols
    
    final_df = merged[keep_cols].copy()
    
    # time_acc1 をドロップ (旧ロジック維持)
    final_df = final_df.drop(columns=["time_acc1"])
    
    # 欠損値の線形補完オプション（旧ロジック維持）
    if interpolate_missing:
        final_df = final_df.interpolate(method='linear', axis=0).ffill().bfill()
        print("✔ 欠損値を補完しました（interpolate）")

    # 最初の450データを落とす（旧ロジック維持）
    final_df = final_df.iloc[450:, :].reset_index(drop=True)
    
    # 標準化（旧ロジック維持）
    sc = StandardScaler()
    final_df = pd.DataFrame(sc.fit_transform(final_df), columns=final_df.columns)

    # 保存
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        final_df.to_csv(save_path, index=False)
        print("✔ 保存:", save_path)

    return final_df

In [16]:
df = load_and_process_by_keyword("kannno1",save_path="data2/kannno1_data.csv")
# df = load_and_process_by_keyword("kannno1")


------- kannno1 -------
✔ acc1 → kannno1_AppTag_10D739C_ADXL34x_20251116.csv
✔ acc2 → kannno1_AppTag_10DA3D9_ADXL34x_20251116.csv
✔ beacon → kannno1.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/kannno1_data.csv


In [17]:
df.head()

,acc_x_acc1,acc_y_acc1,acc_z_acc1,acc_mag_acc1,acc_mag_diff_acc1,acc_x_diff_acc1,acc_y_diff_acc1,acc_z_diff_acc1,acc_x_band1_power_win_0.2-0.7Hz_acc1,acc_y_band1_power_win_0.2-0.7Hz_acc1,...,81025B89_max,810B3B76_max,810257F7_min,81025919_min,81025B89_min,810B3B76_min,810257F7_range,81025919_range,81025B89_range,810B3B76_range
0,0.574624,0.109457,-0.946619,-0.346664,-0.151228,-0.031158,-0.081357,-0.203883,-0.531473,1.545257,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
1,0.574624,0.184964,-0.961175,-0.350129,-0.007796,-0.000012,0.244870,-0.033944,-0.531538,1.544402,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
2,0.582944,0.210133,-0.932063,-0.315278,0.078504,0.031134,0.081756,0.068020,-0.531603,1.543496,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
3,0.582944,0.059119,-0.801059,-0.249917,0.039087,0.124574,0.081756,-0.203883,-0.531344,1.546816,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
4,0.582944,0.134626,-0.859283,-0.279516,-0.066657,-0.000012,0.244870,-0.135907,-0.531409,1.546061,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767


In [18]:
# df.iloc[:,64:].describe()
df.iloc[:,64:].head()

,810257F7,81025919,81025B89,810B3B76,810257F7_mean,81025919_mean,81025B89_mean,810B3B76_mean,810257F7_std,81025919_std,...,81025B89_max,810B3B76_max,810257F7_min,81025919_min,81025B89_min,810B3B76_min,810257F7_range,81025919_range,81025B89_range,810B3B76_range
0,-1.052162,-1.487675,-0.85599,2.46917,-1.822954,-2.097931,1.657757,0.923397,-1.914141,0.24618,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
1,-1.052162,-1.487675,-0.85599,2.46917,-1.822954,-2.097931,1.657757,0.923397,-1.914141,0.24618,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
2,-1.052162,-1.487675,-0.85599,2.46917,-1.822954,-2.097931,1.657757,0.923397,-1.914141,0.24618,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
3,-1.052162,-1.487675,-0.85599,2.46917,-1.822954,-2.097931,1.657757,0.923397,-1.914141,0.24618,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767
4,-1.052162,-1.487675,-0.85599,2.46917,-1.822954,-2.097931,1.657757,0.923397,-1.914141,0.24618,...,0.274678,0.679092,-0.713609,-0.774114,0.85238,1.665168,-2.055159,-1.505191,-0.146074,-0.338767


In [19]:
keywords = ['kannno1','kannno2', 'sato1', 'sato2', 'nishi2', 'morita1','morita2','mori1','mori2']
for kw in keywords:
    save_path = f"data2/{kw}_data.csv"
    df = load_and_process_by_keyword(kw, save_path=save_path)
    

------- kannno1 -------
✔ acc1 → kannno1_AppTag_10D739C_ADXL34x_20251116.csv
✔ acc2 → kannno1_AppTag_10DA3D9_ADXL34x_20251116.csv
✔ beacon → kannno1.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/kannno1_data.csv
------- kannno2 -------
✔ acc1 → kannno2_AppTag_10DA3D9_ADXL34x_20251116.csv
✔ acc2 → kannno2_AppTag_10D739C_ADXL34x_20251116.csv
✔ beacon → kannno2.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/kannno2_data.csv
------- sato1 -------
✔ acc1 → sato1_AppTag_10D739C_ADXL34x_20251116.csv
✔ acc2 → sato1_AppTag_10DA3D9_ADXL34x_20251116.csv
✔ beacon → sato1.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/sato1_data.csv
------- sato2 -------
✔ acc1 → sato2_AppTag_10DA3D9_ADXL34x_20251116.csv
✔ acc2 → sato2_AppTag_10D739C_ADXL34x_20251116.csv
✔ beacon → sato2.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/sato2_data.csv
------- nishi2 -------
✔ acc1 → nishi2_AppTag_10D739C_ADXL34x_20251122.csv
✔ acc2 → nishi2_AppTag_10DA3D9_ADXL34x_20251122.csv
✔ beacon → nishi2.csv
✔ 欠損値を補完しました（interpolate）
✔ 保存: data2/n